# 04 · Descriptive training — Gemma 3 270M + TableViT-Lite + LoRA

End-to-end training driven by `configs/config.yaml` and the DataLoader from
notebook 02. Everything the spec demands is wired in:

| requirement | where |
|---|---|
| train/val split | two manifests from notebook 01 |
| resume training | `training.resume_from` → `CheckpointManager.resume` (optimizer, scheduler, RNG, counters) |
| descriptive losses (train & val) | tqdm postfix + `logs/training.log` + `logs/metrics.jsonl` |
| step / epoch info, optimizer, LR | logged every `log_every_steps` |
| **logging skipped steps** | NaN/Inf loss or grad-norm → step skipped, counted & logged |
| save weights per epoch | `last.pt` (always) |
| save **only if better** | `best.pt` written iff val loss improved |
| tqdm indicators | train + validation bars |

**Curriculum (recommended):** run once with `training.stage: align`
(LINEARIZE targets, Gemma fully frozen → projector/vision warm-up), then set
`stage: sft` and run again (LoRA on, JSON targets). The `sft` run can
`resume_from` nothing — it loads the aligned vision/projector via the resume
cell below if you point it at the align `best.pt`.

In [ ]:
# --- bootstrap: make the src/ package importable from notebooks/ ---
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)  # so relative paths in configs/config.yaml resolve

from gemma_ft_json.config import load_config
cfg = load_config("configs/config.yaml")
print("config loaded; data_root =", cfg.paths.data_root)


## 1. Device & precision — MPS-first

In [ ]:
from gemma_ft_json.utils import resolve_device, resolve_dtype, set_seed
set_seed(cfg.training.seed)
device = resolve_device(cfg.device.preferred, cfg.device.mps_fallback_env)
dtype = resolve_dtype(cfg.device.dtype)
print("device:", device, "| dtype:", dtype)   # Mac M4 → mps

## 2. Load LOCAL Gemma (strictly offline) and fuse the vision pathway

In [ ]:
from gemma_ft_json.models import load_gemma_local, GemmaVisionForJSON

gemma, tok = load_gemma_local(cfg.paths.gemma_model_dir, dtype)
model = GemmaVisionForJSON(gemma, tok, cfg.model,
                           image_size=cfg.dataset.image_size,
                           stage=cfg.training.stage)
print("LM hidden:", model.lm_dim, "| visual tokens:", model.num_visual_tokens)
print("LoRA injected into:", len(model.lora_paths), "linears")
print("projector RMS target:", float(model.projector.target_rms))  # ≈ sqrt(640)≈25

## 3. DataLoaders (train + val) from the notebook-02 recipe

In [ ]:
from torch.utils.data import DataLoader
from gemma_ft_json.data import TableImageJsonDataset, VLMCollator

stage = cfg.training.stage if cfg.training.stage != "align" else "linearize"
def make(manifest, shuffle):
    ds = TableImageJsonDataset(manifest, tok, cfg.dataset.image_size,
                               stage=stage, max_seq_len=cfg.model.max_seq_len)
    return DataLoader(ds, batch_size=cfg.training.batch_size, shuffle=shuffle,
                      num_workers=cfg.training.num_workers,
                      collate_fn=VLMCollator(tok.pad_token_id,
                                             cfg.training.structure_loss_weight,
                                             cfg.training.content_loss_weight),
                      persistent_workers=cfg.training.num_workers > 0)
train_loader, val_loader = make(cfg.paths.manifest_train, True), make(cfg.paths.manifest_val, False)
print(len(train_loader.dataset), "train /", len(val_loader.dataset), "val samples")

## 4. (Optional) resume
Set a path to continue an interrupted run — restores model, optimizer,
scheduler, RNG state, epoch and step counters, and the best-val watermark.

In [ ]:
# cfg.training.resume_from = str(REPO / "checkpoints" / "last.pt")   # ← uncomment to resume
print("resume_from =", cfg.training.resume_from)

## 5. Train

In [ ]:
from gemma_ft_json.training import Trainer
trainer = Trainer(model, train_loader, val_loader, cfg, device)
result = trainer.train()
result

## 6. What to watch
* `loss` in the tqdm bar should fall fast for ~200 steps (structure tokens),
  then grind slowly (content tokens — the real learning).
* `skips` should stay at **0**; a non-zero count is logged with reasons in
  `logs/training.log` and `logs/metrics.jsonl` (`event: "skip"`).
* `best.pt` only updates on val improvement — the log prints
  `>> new best.pt <<` vs `no improvement, best.pt unchanged` per epoch.
* Open **notebook 05 in parallel** for live curves while this runs.